<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Third_Places.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.0 MB/s eta 0:00:00


In [7]:
!wget https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
!unzip latinobarometro-2024-stata-v20250817.zip

--2026-05-15 20:13:00--  https://www.latinobarometro.org/documents/LAT-2024/latinobarometro-2024-stata-v20250817.zip
Resolving www.latinobarometro.org (www.latinobarometro.org)... 80.28.53.190
Connecting to www.latinobarometro.org (www.latinobarometro.org)|80.28.53.190|:443... connected.
HTTP request sent, awaiting response... 200 200
Length: 6691592 (6.4M) [application/x-zip-compressed]
Saving to: ‘latinobarometro-2024-stata-v20250817.zip’

latinobarometro-202 100%[===================>]   6.38M  6.67MB/s    in 1.0s    

2026-05-15 20:13:02 (6.67 MB/s) - ‘latinobarometro-2024-stata-v20250817.zip’ saved [6691592/6691592]

Archive:  latinobarometro-2024-stata-v20250817.zip
  inflating: Latinobarometro_2024_Stata_eng_v20250817.dta  
  inflating: Latinobarometro_2024_Stata_esp_v20250817.dta  
  inflating: Latinobarometro_2024_Cuestionario_esp.pdf  
  inflating: Latinobarometro_2024_Cuestionario_eng.pdf  


In [16]:
import pandas as pd

# abrir reader
reader = pd.read_stata(
    "Latinobarometro_2024_Stata_esp_v20250817.dta",
    convert_categoricals=False,
    iterator=True
)

# dataframe
df24 = reader.read()

# labels de variables
variable_labels = reader.variable_labels()

# TODOS los value labels
value_labels = reader.value_labels()

In [6]:
import osmnx as ox

# =========================
# CONFIGURACIÓN
# =========================

place = "Rosario, Santa Fe, Argentina"
population = 1300000  # cambiar por la población de la ciudad

# =========================
# DESCARGAR POLÍGONO
# =========================

gdf = ox.geocode_to_gdf(place)
polygon = gdf.geometry.iloc[0]

# =========================
# DEFINIR "THIRD PLACES"
# =========================

tags = {
    "amenity": [
        "cafe",
        "bar",
        "pub",
        "library",
        "community_centre",
        "place_of_worship"
    ],
    "leisure": [
        "park",
        "sports_centre"
    ]
}

# =========================
# DESCARGAR POIs
# =========================

pois = ox.features_from_polygon(polygon, tags)

# eliminar duplicados geométricos
pois = pois.drop_duplicates(subset=["geometry"])

# =========================
# INDICADOR FINAL
# =========================

third_places_per_100k = (
    len(pois) / population
) * 100000

print(round(third_places_per_100k, 2))

93.46


In [21]:
import pandas as pd
import numpy as np
import osmnx as ox
from shapely.geometry import Point

# =========================================================
# LABELS
# =========================================================

pais_labels = value_labels["IDENPA"]
ciudad_labels = value_labels["CIUDAD"]

# =========================================================
# UNIQUE CITIES
# =========================================================

cities = (
    df24[["IDENPA", "CIUDAD", "TAMCIUD"]]
    .dropna()
    .drop_duplicates(subset=["IDENPA", "CIUDAD"])
    .copy()
)

cities["country"] = cities["IDENPA"].map(pais_labels)
cities["city"] = cities["CIUDAD"].map(ciudad_labels)

# limpiar
cities["country"] = (
    cities["country"]
    .astype(str)
    .str.replace(r"\[%\d+%\]\s*", "", regex=True)
    .str.strip()
)

cities["city"] = (
    cities["city"]
    .astype(str)
    .str.replace(r"^[A-Z]{2}:\s*", "", regex=True)
    .str.strip()
)

# =========================================================
# TAGS
# =========================================================

tags = {
    "amenity": [
        "cafe",
        "bar",
        "pub",
        "library",
        "community_centre",
        "place_of_worship"
    ],
    "leisure": [
        "park",
        "sports_centre"
    ]
}

# =========================================================
# LOOP
# =========================================================

results = []

for _, row in cities.iterrows():

    city = row["city"]
    country = row["country"]
    tam = row["TAMCIUD"]

    # query original
    query = f"{city}, {country}"

    try:

        try:
            # intento normal
            gdf = ox.geocode_to_gdf(query)

        except:

            # fallback:
            # usar parte después del "-"
            if "-" in city:
                fallback_city = city.split("-")[-1].strip()
            else:
                fallback_city = city

            fallback_query = f"{fallback_city}, {country}"

            gdf = ox.geocode_to_gdf(fallback_query)

        # usar polígono si existe
        polygon = gdf.geometry.iloc[0]

        # descargar POIs
        pois = ox.features_from_polygon(polygon, tags)

        # si no encuentra nada, intentar buffer circular
        if len(pois) == 0:

            centroid = polygon.centroid

            # buffer ~5km
            buffer = centroid.buffer(0.05)

            pois = ox.features_from_polygon(buffer, tags)

        pois = pois.drop_duplicates(subset=["geometry"])

        n_places = len(pois)

        adjusted_index = n_places / np.log(tam + 1)

        results.append({
            "country": country,
            "city": city,
            "tamciud": tam,
            "third_places": n_places,
            "third_places_adj": adjusted_index
        })

        print(f"OK: {query} -> {n_places}")

    except Exception as e:

        print(f"ERROR: {query} -> {e}")

# =========================================================
# FINAL DF
# =========================================================

df_third_places = pd.DataFrame(results)

print(df_third_places.head())

ERROR: Buenos Aires-Gran Buenos Aires, Argentina -> Nominatim did not geocode query 'Gran Buenos Aires, Argentina' to a geometry of type (Multi)Polygon.
ERROR: Buenos Aires-La Matanza, Argentina -> No matching features. Check query location, tags, and log.
OK: Buenos Aires-Lomas de Zamora, Argentina -> 82
OK: Buenos Aires-Quilmes, Argentina -> 3
ERROR: Buenos Aires-Lanus, Argentina -> No matching features. Check query location, tags, and log.
ERROR: Buenos Aires-Almirante Brown, Argentina -> No matching features. Check query location, tags, and log.
OK: Buenos Aires-Merlo, Argentina -> 1
ERROR: Buenos Aires-General San Martin, Argentina -> No matching features. Check query location, tags, and log.
ERROR: Buenos Aires-Avellaneda, Argentina -> No matching features. Check query location, tags, and log.
OK: Buenos Aires-San Isidro, Argentina -> 1
ERROR: Buenos Aires-Tres de Febrero, Argentina -> No matching features. Check query location, tags, and log.
OK: Buenos Aires-Moron, Argentina ->

/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 58 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


KeyboardInterrupt: 